# 01 — Feature Engineering (Local)

Reimplements `databricks/notebooks/processing/03_gold_features.py` in pure pandas.
Load sample CSV fixtures, compute rolling features and competitive signals, and
validate the output matches the expected gold schema.

**Why this exists:** Iterate on feature logic locally (no cluster startup) before
updating the Databricks production notebook.

In [ ]:
import sys
import os

# Add databricks/src/ to path so we can import shared modules
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "../.."))
SRC_ROOT  = os.path.join(REPO_ROOT, "databricks")
if SRC_ROOT not in sys.path:
    sys.path.insert(0, SRC_ROOT)

import numpy as np
import pandas as pd

from src.features.definitions import (
    PRICE_POSITION_BAND,
    RATING_PREMIUM_PCT,
    RATING_PREMIUM_THRESHOLD,
    ROLLING_WINDOW_DAYS,
)
from src.features.validation import check_required_columns, check_minimum_rows

DATA_DIR = os.path.join(os.getcwd(), "../data")
print(f"Repo root : {REPO_ROOT}")
print(f"Data dir  : {DATA_DIR}")

## Load sample data

In [ ]:
price_hist = pd.read_csv(os.path.join(DATA_DIR, "sample_price_history.csv"), parse_dates=["scrape_date"])
comp_map   = pd.read_csv(os.path.join(DATA_DIR, "sample_competitor_map.csv"), parse_dates=["scrape_date"])

print(f"Price history : {price_hist.shape}")
print(f"Competitor map: {comp_map.shape}")
price_hist.head()

## Rolling time-series features

Mirror of `03_gold_features.py` lines 59–75.
`ROLLING_WINDOW_DAYS` is read from `src.features.definitions` — same constant as production.

In [ ]:
own = price_hist[price_hist["source"] == "product"].copy().sort_values(["asin", "scrape_date"])

own["price_7d_ma"] = (
    own.groupby("asin")["price"]
    .transform(lambda x: x.rolling(ROLLING_WINDOW_DAYS, min_periods=1).mean())
)
own["price_7d_volatility"] = (
    own.groupby("asin")["price"]
    .transform(lambda x: x.rolling(ROLLING_WINDOW_DAYS, min_periods=1).std())
)
own["price_momentum"] = (
    own.groupby("asin")["price"]
    .transform(lambda x: x.pct_change(ROLLING_WINDOW_DAYS))
)

print(f"Rolling window: {ROLLING_WINDOW_DAYS} days")
own[["asin", "scrape_date", "price", "price_7d_ma", "price_7d_volatility", "price_momentum"]].tail(10)

## Competitive aggregates

In [ ]:
comp_agg = (
    comp_map.groupby(["parent_asin", "scrape_date"])["price"]
    .agg(
        comp_median_price="median",
        comp_avg_price="mean",
        comp_min_price="min",
        comp_max_price="max",
        comp_count="count",
    )
    .reset_index()
    .rename(columns={"parent_asin": "asin"})
)

gold = own.merge(comp_agg, on=["asin", "scrape_date"], how="left")
print(f"Gold rows: {len(gold)}")
gold[["asin", "scrape_date", "price", "comp_median_price", "comp_count"]].tail(5)

## Competitors below + pressure score

In [ ]:
def count_below(row):
    mask = (
        (comp_map["parent_asin"] == row["asin"]) &
        (comp_map["scrape_date"] == row["scrape_date"]) &
        (comp_map["price"] < row["price"])
    )
    return int(comp_map[mask].shape[0])

gold["competitors_below"] = gold.apply(count_below, axis=1)
gold["comp_pressure_score"] = gold["competitors_below"] / gold["comp_count"].replace(0, np.nan)

gold[["asin", "scrape_date", "comp_count", "competitors_below", "comp_pressure_score"]].tail(5)

## Suggested price + price position

Constants come from `src.features.definitions` — identical to what production uses.

In [ ]:
comp_avg_rating = (
    comp_map.groupby(["parent_asin", "scrape_date"])["rating"]
    .mean()
    .reset_index()
    .rename(columns={"parent_asin": "asin", "rating": "comp_avg_rating"})
)
gold = gold.merge(comp_avg_rating, on=["asin", "scrape_date"], how="left")

premium = np.where(
    (gold["rating"] - gold["comp_avg_rating"]) > RATING_PREMIUM_THRESHOLD,
    RATING_PREMIUM_PCT, 0.0
)
gold["suggested_price"] = (gold["comp_median_price"] * (1 + premium)).round(2)

gold["price_position"] = np.where(
    gold["comp_median_price"].isna(), None,
    np.where(gold["price"] > gold["comp_median_price"] * (1 + PRICE_POSITION_BAND), "above_market",
    np.where(gold["price"] < gold["comp_median_price"] * (1 - PRICE_POSITION_BAND), "below_market",
             "at_market"))
)

print(f"Rating premium threshold : {RATING_PREMIUM_THRESHOLD} stars → +{RATING_PREMIUM_PCT*100:.0f}%")
print(f"Price position band      : ±{PRICE_POSITION_BAND*100:.0f}%")
gold[["asin", "scrape_date", "price", "suggested_price", "price_position"]].tail(5)

## Validate schema

In [ ]:
from src.features.definitions import FEATURE_COLS, TARGET_COL

gold = gold.rename(columns={"price": "current_price"})

check_required_columns(gold, FEATURE_COLS + [TARGET_COL])
check_minimum_rows(gold, 10)

print("✓ Schema validation passed")
print(f"\nNull rates in feature columns:")
print(gold[FEATURE_COLS + [TARGET_COL]].isna().mean().round(3).to_string())

## Compare with pre-generated gold features

In [ ]:
gold_ref = pd.read_csv(os.path.join(DATA_DIR, "sample_gold_features.csv"), parse_dates=["scrape_date"])

# Compare suggested_price distributions
print("Our computed suggested_price:")
print(gold["suggested_price"].describe().round(2))
print("\nReference (generate_sample_data.py):")
print(gold_ref["suggested_price"].describe().round(2))
print("\nprice_position breakdown:")
print(gold["price_position"].value_counts())